# 🏆 Golden Dataset Creator

**Amaç:** 1.9M resimden kaliteli, çeşitli ve duplicate-free bir "Golden Dataset" oluştur

**Pipeline:**
1. 🔄 **Deduplication** - Near-duplicate temizliği (FAISS + Cosine Similarity > 0.98)
2. 🎯 **K-Means Clustering** - Ana konseptleri bul ve temsili örnekler seç
3. 🌟 **Farthest Point Sampling** - Edge case ve outlier'ları yakala
4. 📊 **UMAP Görselleştirme** - Seçimi doğrula

**Giriş:** `all_embeddings.parquet` (folder_name, image_name, embedding)
**Çıkış:** `golden_dataset.parquet` (folder_name, image_name, embedding, selection_method)

In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
# 🔗 Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 📦 Gerekli paketleri yükle
!pip install -q faiss-gpu-cu12 pandas pyarrow scikit-learn umap-learn plotly tqdm

In [ ]:
# ⚙️ AYARLAR
import os
from pathlib import Path

# Embedding dosyası (Drive)
PARQUET_FILE = f"{DRIVE_ROOT}/embeddings/all_embeddings.parquet"

# Çıktı klasörü
OUTPUT_DIR = f"{DRIVE_ROOT}/golden_dataset"

# === Deduplication Ayarları ===
DUPLICATE_THRESHOLD = 0.98  # Cosine similarity > bu değer = duplicate

# === K-Means Ayarları ===
N_CLUSTERS = 5000  # Hedef cluster sayısı (bütçene göre ayarla)
KMEANS_BATCH_SIZE = 4096  # MiniBatchKMeans için

# === Seçim Oranları ===
KMEANS_RATIO = 0.7  # %70 K-Means'ten
FPS_RATIO = 0.3     # %30 Farthest Point Sampling'den

# === Hedef Dataset Boyutu ===
TARGET_SIZE = 40000  # Golden dataset'te kaç resim olsun?

# Klasör oluştur
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Giriş: {PARQUET_FILE}")
print(f"📁 Çıkış: {OUTPUT_DIR}")
print(f"🎯 Hedef boyut: {TARGET_SIZE:,}")
print(f"   - K-Means: {int(TARGET_SIZE * KMEANS_RATIO):,} (%{int(KMEANS_RATIO*100)})")
print(f"   - FPS: {int(TARGET_SIZE * FPS_RATIO):,} (%{int(FPS_RATIO*100)})")

In [ ]:
# 📚 Import'lar
import numpy as np
import pandas as pd
import faiss
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances_argmin_min
from tqdm.notebook import tqdm
import json
import gc

# GPU kontrolü
import torch
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"💾 GPU: {torch.cuda.get_device_name(0)}")
    
# FAISS GPU kontrolü
try:
    res = faiss.StandardGpuResources()
    print("✅ FAISS GPU desteği aktif")
    USE_GPU = True
except:
    print("⚠️ FAISS GPU yok, CPU kullanılacak")
    USE_GPU = False

In [ ]:
# 📥 Veriyi yükle
print("📥 Parquet dosyası yükleniyor...")
df = pd.read_parquet(PARQUET_FILE)

print(f"📊 Yüklenen veri:")
print(f"   Toplam resim: {len(df):,}")
print(f"   Kolonlar: {list(df.columns)}")

# Embedding'leri numpy array'e çevir
print("\n🔄 Embedding'ler numpy array'e dönüştürülüyor...")
embeddings = np.array(df['embedding'].tolist(), dtype=np.float32)
print(f"   Shape: {embeddings.shape}")
print(f"   Memory: {embeddings.nbytes / 1e9:.2f} GB")

# L2 normalize et (cosine similarity için gerekli)
print("\n🔄 L2 normalizasyon...")
faiss.normalize_L2(embeddings)
print("   ✅ Normalizasyon tamamlandı")

---
## 🔄 Adım 1: Deduplication (Near-Duplicate Temizliği)

FAISS ile her embedding için en yakın komşuları bul. Cosine similarity > 0.98 olanları grupla ve her gruptan sadece birini tut.

In [ ]:
# 🔄 Deduplication fonksiyonu
def deduplicate_embeddings(embeddings: np.ndarray, threshold: float = 0.98, batch_size: int = 10000):
    """
    FAISS ile near-duplicate tespiti ve temizleme.
    
    Args:
        embeddings: L2 normalized embedding array (N, 512)
        threshold: Cosine similarity eşiği (0.98 = çok benzer)
        batch_size: FAISS arama batch boyutu
    
    Returns:
        keep_indices: Tutulacak unique resimlerin indexleri
    """
    n_samples, dim = embeddings.shape
    print(f"🔄 Deduplication başlıyor...")
    print(f"   Toplam: {n_samples:,} resim")
    print(f"   Threshold: {threshold}")
    
    # FAISS index oluştur (Inner Product = Cosine Similarity for normalized vectors)
    if USE_GPU:
        # GPU index
        res = faiss.StandardGpuResources()
        flat_config = faiss.GpuIndexFlatConfig()
        flat_config.device = 0
        index = faiss.GpuIndexFlatIP(res, dim, flat_config)
    else:
        # CPU index
        index = faiss.IndexFlatIP(dim)
    
    # Tüm vektörleri indexe ekle
    print("   📦 FAISS index oluşturuluyor...")
    index.add(embeddings)
    
    # Her vektör için en yakın K komşuyu bul
    # K=10 yeterli - genelde duplicate'ler çok yakın
    K = 10
    
    # Union-Find yapısı ile grupları takip et
    parent = list(range(n_samples))
    
    def find(x):
        if parent[x] != x:
            parent[x] = find(parent[x])
        return parent[x]
    
    def union(x, y):
        px, py = find(x), find(y)
        if px != py:
            parent[px] = py
    
    # Batch halinde arama yap
    print("   🔍 Near-duplicate arama...")
    for start in tqdm(range(0, n_samples, batch_size), desc="Dedup batches"):
        end = min(start + batch_size, n_samples)
        batch = embeddings[start:end]
        
        # En yakın K komşuyu bul
        similarities, indices = index.search(batch, K)
        
        # Threshold üstündeki benzerlikleri grupla
        for i, (sims, idxs) in enumerate(zip(similarities, indices)):
            global_idx = start + i
            for sim, neighbor_idx in zip(sims, idxs):
                if neighbor_idx != global_idx and sim >= threshold:
                    union(global_idx, neighbor_idx)
    
    # Her gruptan en küçük index'i tut (ilk eklenen)
    print("   📊 Gruplar hesaplanıyor...")
    groups = {}
    for i in range(n_samples):
        root = find(i)
        if root not in groups:
            groups[root] = []
        groups[root].append(i)
    
    # Her gruptan birini seç
    keep_indices = [min(group) for group in groups.values()]
    keep_indices.sort()
    
    n_duplicates = n_samples - len(keep_indices)
    n_groups = len(groups)
    
    print(f"\n📊 Deduplication sonucu:")
    print(f"   Orijinal: {n_samples:,}")
    print(f"   Duplicate grupları: {n_groups:,}")
    print(f"   Kaldırılan: {n_duplicates:,} ({n_duplicates/n_samples*100:.1f}%)")
    print(f"   Kalan: {len(keep_indices):,}")
    
    return np.array(keep_indices)

print("✅ Deduplication fonksiyonu yüklendi")

In [ ]:
# ▶️ Deduplication çalıştır
unique_indices = deduplicate_embeddings(embeddings, threshold=DUPLICATE_THRESHOLD)

# Deduplicated veriyi hazırla
df_unique = df.iloc[unique_indices].reset_index(drop=True)
embeddings_unique = embeddings[unique_indices]

print(f"\n✅ Deduplicated data hazır:")
print(f"   DataFrame: {len(df_unique):,} satır")
print(f"   Embeddings: {embeddings_unique.shape}")

---
## 🔍 Adım 1.5: Kalite Filtresi (L2 Norm)

CLIP embedding'lerinde düşük L2 norm = düşük kaliteli/bozuk resim.
Normalizasyondan ÖNCE orijinal norm değerlerine bakarak kalitesiz resimleri filtrele.

In [ ]:
# 🔍 Kalite Filtresi - L2 Norm Analizi
# ⚠️ Bu adım normalizasyondan ÖNCE yapılmalı!
# Eğer zaten normalize ettiysen, veriyi tekrar yüklemen gerekir

def analyze_and_filter_by_quality(df: pd.DataFrame, embeddings: np.ndarray, 
                                   low_percentile: float = 1.0, 
                                   high_percentile: float = 99.5):
    """
    L2 norm'a göre kalite analizi ve filtreleme.
    
    CLIP'te:
    - Çok düşük norm → bozuk/yarım yüklenmiş/boş resimler
    - Çok yüksek norm → nadir, genelde sorunlu değil
    - Normal aralık → kaliteli resimler
    
    Args:
        df: DataFrame with embeddings
        embeddings: Raw (unnormalized) embeddings
        low_percentile: Alt kesim yüzdesi (default: alt %1)
        high_percentile: Üst kesim yüzdesi (default: üst %0.5)
    
    Returns:
        keep_indices: Kaliteli resimlerin indexleri
        stats: İstatistikler
    """
    print("🔍 Kalite analizi başlıyor (L2 Norm)...")
    
    # L2 norm hesapla
    norms = np.linalg.norm(embeddings, axis=1)
    
    print(f"\n📊 L2 Norm İstatistikleri:")
    print(f"   Min:    {norms.min():.4f}")
    print(f"   Max:    {norms.max():.4f}")
    print(f"   Mean:   {norms.mean():.4f}")
    print(f"   Std:    {norms.std():.4f}")
    print(f"   Median: {np.median(norms):.4f}")
    
    # Percentile'lar
    percentiles = [0.1, 1, 5, 10, 25, 50, 75, 90, 95, 99, 99.9]
    print(f"\n📈 Percentile dağılımı:")
    for p in percentiles:
        print(f"   P{p:5.1f}: {np.percentile(norms, p):.4f}")
    
    # Filtreleme eşikleri
    low_threshold = np.percentile(norms, low_percentile)
    high_threshold = np.percentile(norms, high_percentile)
    
    print(f"\n🎯 Filtreleme eşikleri:")
    print(f"   Alt eşik (P{low_percentile}): {low_threshold:.4f}")
    print(f"   Üst eşik (P{high_percentile}): {high_threshold:.4f}")
    
    # Filtreleme
    low_quality_mask = norms < low_threshold
    high_quality_mask = norms > high_threshold
    keep_mask = ~low_quality_mask & ~high_quality_mask
    
    n_low = low_quality_mask.sum()
    n_high = high_quality_mask.sum()
    n_keep = keep_mask.sum()
    
    print(f"\n📊 Filtreleme sonucu:")
    print(f"   ❌ Düşük kalite (norm < {low_threshold:.4f}): {n_low:,} ({n_low/len(norms)*100:.2f}%)")
    print(f"   ❌ Çok yüksek norm (norm > {high_threshold:.4f}): {n_high:,} ({n_high/len(norms)*100:.2f}%)")
    print(f"   ✅ Kaliteli: {n_keep:,} ({n_keep/len(norms)*100:.2f}%)")
    
    # Düşük kaliteli örnekleri göster
    if n_low > 0:
        print(f"\n🔍 En düşük norm'lu 10 resim (potansiyel bozuk):")
        lowest_indices = np.argsort(norms)[:10]
        for idx in lowest_indices:
            row = df.iloc[idx]
            print(f"   norm={norms[idx]:.4f} | {row['folder_name']}/{row['image_name'][:50]}")
    
    keep_indices = np.where(keep_mask)[0]
    
    stats = {
        "total": len(norms),
        "filtered_low": int(n_low),
        "filtered_high": int(n_high),
        "kept": int(n_keep),
        "low_threshold": float(low_threshold),
        "high_threshold": float(high_threshold),
        "mean_norm": float(norms.mean()),
        "std_norm": float(norms.std())
    }
    
    return keep_indices, stats, norms

print("✅ Kalite filtresi fonksiyonu yüklendi")

In [ ]:
# ▶️ Kalite filtresini çalıştır
# ⚠️ ÖNEMLİ: Bu adım için embeddings'i TEKRAR yüklemen gerekir (normalize edilmemiş hali)
# Çünkü L2 normalize edilmiş vektörlerin hepsinin norm'u 1.0 olur

print("⚠️ Embeddings'i normalize edilmemiş haliyle yüklüyoruz...")
embeddings_raw = np.array(df_unique['embedding'].tolist(), dtype=np.float32)
print(f"   Shape: {embeddings_raw.shape}")

# Kalite filtresi uygula
quality_indices, quality_stats, norms = analyze_and_filter_by_quality(
    df_unique, 
    embeddings_raw,
    low_percentile=1.0,   # Alt %1'i filtrele
    high_percentile=99.9  # Üst %0.1'i filtrele (çok az)
)

# Kaliteli veriyi hazırla
df_quality = df_unique.iloc[quality_indices].reset_index(drop=True)
embeddings_quality = embeddings_unique[quality_indices]  # Normalize edilmiş hali kullan

print(f"\n✅ Kalite filtresi uygulandı:")
print(f"   Giriş: {len(df_unique):,} resim")
print(f"   Çıkış: {len(df_quality):,} resim")
print(f"   Filtrelenen: {len(df_unique) - len(df_quality):,} resim")

---
## 🎯 Adım 2: K-Means Clustering

Embedding uzayını K cluster'a böl. Her cluster'ın merkezine en yakın resmi seç = en temsili örnekler.

In [ ]:
# 🎯 K-Means Clustering
def select_by_kmeans(embeddings: np.ndarray, n_select: int, n_clusters: int = None, batch_size: int = 4096):
    """
    K-Means ile temsili örnekleri seç.
    
    Args:
        embeddings: Embedding array (N, 512)
        n_select: Seçilecek örnek sayısı
        n_clusters: Cluster sayısı (default = n_select)
        batch_size: MiniBatchKMeans batch boyutu
    
    Returns:
        selected_indices: Seçilen örneklerin indexleri
        cluster_labels: Her örneğin cluster'ı
    """
    if n_clusters is None:
        n_clusters = n_select
    
    n_samples = len(embeddings)
    
    # n_clusters, n_samples'dan büyük olamaz
    n_clusters = min(n_clusters, n_samples)
    n_select = min(n_select, n_samples)
    
    print(f"🎯 K-Means Clustering başlıyor...")
    print(f"   Veri: {n_samples:,} resim")
    print(f"   Cluster sayısı: {n_clusters:,}")
    print(f"   Seçilecek: {n_select:,}")
    
    # MiniBatchKMeans - büyük veri için hızlı
    print("   ⏳ Clustering...")
    kmeans = MiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=batch_size,
        n_init=3,
        max_iter=100,
        random_state=42,
        verbose=0
    )
    
    cluster_labels = kmeans.fit_predict(embeddings)
    centers = kmeans.cluster_centers_
    
    print("   ✅ Clustering tamamlandı")
    
    # Her cluster merkezine en yakın örneği bul
    print("   🔍 Merkeze en yakın örnekler bulunuyor...")
    closest_indices, distances = pairwise_distances_argmin_min(centers, embeddings)
    
    # n_select kadar seç (cluster sayısından az olabilir)
    if n_select < n_clusters:
        # En az uzaklığa sahip cluster'ları seç
        sorted_idx = np.argsort(distances)[:n_select]
        selected_indices = closest_indices[sorted_idx]
    else:
        selected_indices = closest_indices
    
    print(f"\n📊 K-Means sonucu:")
    print(f"   Seçilen: {len(selected_indices):,} resim")
    
    return selected_indices, cluster_labels

print("✅ K-Means fonksiyonu yüklendi")

In [ ]:
# ▶️ K-Means seçimi
n_kmeans = int(TARGET_SIZE * KMEANS_RATIO)

# ⚠️ K-Means her cluster'dan 1 resim seçer, bu yüzden n_clusters = n_kmeans olmalı
# Eğer N_CLUSTERS farklı tutmak istiyorsan, her cluster'dan birden fazla örnek almak gerekir
actual_n_clusters = n_kmeans  # Her cluster'dan 1 resim = n_kmeans cluster gerekli

print(f"📌 Not: {n_kmeans:,} resim için {actual_n_clusters:,} cluster kullanılacak")
print(f"📊 Veri: {len(embeddings_quality):,} kaliteli resim üzerinde çalışılacak")

kmeans_indices, cluster_labels = select_by_kmeans(
    embeddings_quality,  # Kalite filtrelenmiş veri
    n_select=n_kmeans,
    n_clusters=actual_n_clusters,
    batch_size=KMEANS_BATCH_SIZE
)

print(f"\n✅ K-Means ile {len(kmeans_indices):,} resim seçildi")

---
## 🌟 Adım 3: Farthest Point Sampling (FPS)

Edge case'leri ve outlier'ları yakalamak için embedding uzayında en uzak noktaları seç.
Bu sayede "garip" veya "zor" örnekler de datasete dahil edilir.

In [ ]:
# 🌟 Optimized Farthest Point Sampling (Batch FPS)
def farthest_point_sampling(embeddings: np.ndarray, n_select: int, exclude_indices: set = None, seed_indices: list = None, batch_size: int = 100):
    """
    Batch Farthest Point Sampling - çok daha hızlı!
    Her iterasyonda birden fazla nokta seçerek 100x hızlanma sağlar.
    
    Args:
        embeddings: Embedding array (N, 512)
        n_select: Seçilecek örnek sayısı
        exclude_indices: Hariç tutulacak indexler (örn: K-Means'ten seçilenler)
        seed_indices: Başlangıç noktaları (örn: K-Means seçimleri)
        batch_size: Her iterasyonda seçilecek nokta sayısı (default: 100)
    
    Returns:
        selected_indices: Seçilen örneklerin indexleri
    """
    n_samples = len(embeddings)
    
    if exclude_indices is None:
        exclude_indices = set()
    
    # Seçilebilir indexler
    available = set(range(n_samples)) - exclude_indices
    
    if len(available) < n_select:
        print(f"⚠️ Yeterli örnek yok. {len(available)} mevcut, {n_select} istendi.")
        n_select = len(available)
    
    print(f"🌟 Batch Farthest Point Sampling başlıyor...")
    print(f"   Havuz: {len(available):,} resim")
    print(f"   Seçilecek: {n_select:,}")
    print(f"   Batch size: {batch_size}")
    
    dim = embeddings.shape[1]
    
    selected = []
    available_arr = np.array(list(available))
    
    # Seed noktalarını al
    if seed_indices is not None and len(seed_indices) > 0:
        seed_embeddings = embeddings[seed_indices]
        print(f"   📌 {len(seed_indices):,} seed noktası kullanılacak")
    else:
        seed_embeddings = None
    
    # Min distance array - her available nokta için en yakın seçilmiş noktaya uzaklık
    # Başlangıçta infinity (henüz hiç nokta seçilmedi)
    min_distances = np.full(len(available_arr), np.inf, dtype=np.float32)
    
    # Seed'lerden uzaklıkları hesapla
    if seed_embeddings is not None:
        print("   ⏳ Seed noktalarından uzaklıklar hesaplanıyor...")
        # Batch halinde hesapla (memory için)
        chunk_size = 50000
        for i in tqdm(range(0, len(available_arr), chunk_size), desc="Seed distances"):
            chunk_indices = available_arr[i:i+chunk_size]
            chunk_embeddings = embeddings[chunk_indices]
            
            # Cosine similarity (normalized vectors için inner product)
            # Distance = 1 - similarity (veya -similarity since we want farthest)
            similarities = chunk_embeddings @ seed_embeddings.T  # (chunk, n_seeds)
            max_sim = similarities.max(axis=1)  # En yakın seed'e similarity
            distances = 1 - max_sim  # Distance
            min_distances[i:i+chunk_size] = distances
    
    n_iterations = (n_select + batch_size - 1) // batch_size
    
    print(f"   📊 {n_iterations} iterasyon yapılacak")
    
    for iteration in tqdm(range(n_iterations), desc="Batch FPS"):
        if len(available_arr) == 0:
            break
        
        # Bu iterasyonda kaç nokta seçeceğiz
        remaining = n_select - len(selected)
        current_batch = min(batch_size, remaining, len(available_arr))
        
        if current_batch <= 0:
            break
        
        # En uzak batch_size noktayı seç
        # argsort descending için -distances kullan (en büyük distance = en uzak)
        farthest_local_indices = np.argpartition(-min_distances, current_batch-1)[:current_batch]
        
        # Global index'lere çevir
        farthest_global_indices = available_arr[farthest_local_indices]
        selected.extend(farthest_global_indices.tolist())
        
        # Seçilen noktaları available'dan çıkar
        mask = np.ones(len(available_arr), dtype=bool)
        mask[farthest_local_indices] = False
        available_arr = available_arr[mask]
        min_distances = min_distances[mask]
        
        if len(available_arr) == 0:
            break
        
        # Yeni seçilen noktalardan uzaklıkları güncelle
        new_selected_embeddings = embeddings[farthest_global_indices]
        remaining_embeddings = embeddings[available_arr]
        
        # Yeni seçilenlerle similarity
        new_similarities = remaining_embeddings @ new_selected_embeddings.T  # (remaining, current_batch)
        new_max_sim = new_similarities.max(axis=1)
        new_distances = 1 - new_max_sim
        
        # Min distance'ı güncelle (eski veya yeni, hangisi daha küçükse)
        min_distances = np.minimum(min_distances, new_distances)
    
    print(f"\n📊 Batch FPS sonucu:")
    print(f"   Seçilen: {len(selected):,} resim")
    
    return np.array(selected)

print("✅ Optimized Batch FPS fonksiyonu yüklendi")

In [ ]:
# ▶️ FPS seçimi
n_fps = int(TARGET_SIZE * FPS_RATIO)

# K-Means'ten seçilenleri hariç tut, ama seed olarak kullan
fps_indices = farthest_point_sampling(
    embeddings_quality,  # Kalite filtrelenmiş veri
    n_select=n_fps,
    exclude_indices=set(kmeans_indices),
    seed_indices=list(kmeans_indices),
    batch_size=100  # 100 nokta/iterasyon = hızlı
)

print(f"\n✅ FPS ile {len(fps_indices):,} resim seçildi")

---
## 🏆 Adım 4: Golden Dataset Oluştur

K-Means ve FPS seçimlerini birleştir ve kaydet.

In [ ]:
# 🏆 Golden Dataset oluştur
print("🏆 Golden Dataset oluşturuluyor...")

# Seçimleri birleştir
all_selected = []

# K-Means seçimleri
for idx in kmeans_indices:
    row = df_quality.iloc[idx]  # Kalite filtrelenmiş DataFrame
    all_selected.append({
        'folder_name': row['folder_name'],
        'image_name': row['image_name'],
        'embedding': row['embedding'],
        'selection_method': 'kmeans'
    })

# FPS seçimleri
for idx in fps_indices:
    row = df_quality.iloc[idx]  # Kalite filtrelenmiş DataFrame
    all_selected.append({
        'folder_name': row['folder_name'],
        'image_name': row['image_name'],
        'embedding': row['embedding'],
        'selection_method': 'fps'
    })

# DataFrame oluştur
golden_df = pd.DataFrame(all_selected)

print(f"\n📊 Golden Dataset özeti:")
print(f"   Toplam: {len(golden_df):,}")
print(f"   K-Means: {len(kmeans_indices):,}")
print(f"   FPS: {len(fps_indices):,}")

# Klasör dağılımı
print(f"\n📁 Klasör dağılımı:")
folder_counts = golden_df['folder_name'].value_counts()
for folder, count in sorted(folder_counts.items())[:20]:  # İlk 20
    print(f"   {folder}: {count}")
if len(folder_counts) > 20:
    print(f"   ... ve {len(folder_counts) - 20} klasör daha")

In [ ]:
# 💾 Golden Dataset'i kaydet
output_file = os.path.join(OUTPUT_DIR, "golden_dataset.parquet")
golden_df.to_parquet(output_file, index=False)

file_size = Path(output_file).stat().st_size / 1e6
print(f"💾 Kaydedildi: {output_file}")
print(f"   Boyut: {file_size:.1f} MB")

# Summary JSON kaydet
summary = {
    "total_images": len(golden_df),
    "original_images": len(df),
    "after_dedup": len(df_unique),
    "selection": {
        "kmeans": len(kmeans_indices),
        "fps": len(fps_indices)
    },
    "settings": {
        "duplicate_threshold": DUPLICATE_THRESHOLD,
        "n_clusters": N_CLUSTERS,
        "target_size": TARGET_SIZE,
        "kmeans_ratio": KMEANS_RATIO,
        "fps_ratio": FPS_RATIO
    },
    "folder_distribution": folder_counts.to_dict()
}

summary_file = os.path.join(OUTPUT_DIR, "golden_summary.json")
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"   ✅ {Path(summary_file).name}")

---
## 📊 Adım 5: UMAP Görselleştirme

Seçilen Golden Dataset'in embedding uzayındaki dağılımını görselleştir.
Amaç: Tüm uzaya homojen dağılım olup olmadığını kontrol et.

In [ ]:
# 📊 UMAP Görselleştirme
import umap
import plotly.express as px
import plotly.graph_objects as go

def visualize_selection(embeddings_all: np.ndarray, selected_indices: np.ndarray, 
                        selection_methods: list, sample_background: int = 50000):
    """
    UMAP ile seçim görselleştirmesi.
    
    Args:
        embeddings_all: Tüm embedding'ler (deduplicated)
        selected_indices: Seçilen indexler
        selection_methods: Her seçim için method ('kmeans' veya 'fps')
        sample_background: Arka plan için örneklenecek sayı
    """
    print("📊 UMAP görselleştirme hazırlanıyor...")
    
    n_total = len(embeddings_all)
    n_selected = len(selected_indices)
    
    # Arka plan için random sample
    if n_total > sample_background:
        bg_indices = np.random.choice(n_total, sample_background, replace=False)
    else:
        bg_indices = np.arange(n_total)
    
    # Seçilenleri de ekle
    all_indices = np.unique(np.concatenate([bg_indices, selected_indices]))
    subset_embeddings = embeddings_all[all_indices]
    
    print(f"   UMAP için {len(subset_embeddings):,} nokta")
    
    # UMAP fit
    print("   ⏳ UMAP hesaplanıyor (bu biraz sürebilir)...")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric='cosine',
        random_state=42,
        verbose=True
    )
    
    embedding_2d = reducer.fit_transform(subset_embeddings)
    
    # Index mapping
    index_to_local = {idx: i for i, idx in enumerate(all_indices)}
    
    # Plot data hazırla
    # Background points
    bg_mask = ~np.isin(all_indices, selected_indices)
    bg_x = embedding_2d[bg_mask, 0]
    bg_y = embedding_2d[bg_mask, 1]
    
    # Selected points
    selected_local = [index_to_local[idx] for idx in selected_indices]
    sel_x = embedding_2d[selected_local, 0]
    sel_y = embedding_2d[selected_local, 1]
    
    # Plotly figure
    fig = go.Figure()
    
    # Background (gri, küçük)
    fig.add_trace(go.Scatter(
        x=bg_x, y=bg_y,
        mode='markers',
        marker=dict(size=2, color='lightgray', opacity=0.3),
        name=f'Diğer ({len(bg_x):,})',
        hoverinfo='skip'
    ))
    
    # K-Means seçimleri (mavi)
    kmeans_mask = [m == 'kmeans' for m in selection_methods]
    fig.add_trace(go.Scatter(
        x=sel_x[kmeans_mask], y=sel_y[kmeans_mask],
        mode='markers',
        marker=dict(size=5, color='blue', opacity=0.7),
        name=f'K-Means ({sum(kmeans_mask):,})'
    ))
    
    # FPS seçimleri (kırmızı)
    fps_mask = [m == 'fps' for m in selection_methods]
    fig.add_trace(go.Scatter(
        x=sel_x[fps_mask], y=sel_y[fps_mask],
        mode='markers',
        marker=dict(size=5, color='red', opacity=0.7),
        name=f'FPS ({sum(fps_mask):,})'
    ))
    
    fig.update_layout(
        title='🏆 Golden Dataset - UMAP Görselleştirme',
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        width=1000,
        height=800,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    return fig

print("✅ Görselleştirme fonksiyonu yüklendi")

In [ ]:
# ▶️ UMAP görselleştirmeyi çalıştır
# Seçilen indexleri ve methodları hazırla
all_selected_indices = np.concatenate([kmeans_indices, fps_indices])
all_methods = ['kmeans'] * len(kmeans_indices) + ['fps'] * len(fps_indices)

fig = visualize_selection(
    embeddings_quality,  # Kalite filtrelenmiş veri
    all_selected_indices,
    all_methods,
    sample_background=50000  # Arka plan için 50K sample
)

fig.show()

In [ ]:
# 💾 UMAP plot'u HTML olarak kaydet (interaktif)
html_file = os.path.join(OUTPUT_DIR, "umap_visualization.html")
fig.write_html(html_file)
print(f"✅ İnteraktif görselleştirme kaydedildi: {html_file}")

---
## 📋 Final Özet ve Sonraki Adımlar

In [ ]:
# 📋 Final Özet
print("=" * 60)
print("🏆 GOLDEN DATASET OLUŞTURULDU!")
print("=" * 60)

print(f"\n📊 Pipeline Özeti:")
print(f"   1. Orijinal veri: {len(df):,} resim")
print(f"   2. Deduplication sonrası: {len(df_unique):,} resim")
print(f"      → Kaldırılan duplicate: {len(df) - len(df_unique):,}")
print(f"   3. Kalite filtresi sonrası: {len(df_quality):,} resim")
print(f"      → Düşük kalite: {len(df_unique) - len(df_quality):,}")
print(f"   4. K-Means seçimi: {len(kmeans_indices):,} resim")
print(f"   5. FPS seçimi: {len(fps_indices):,} resim")
print(f"   6. Final Golden Dataset: {len(golden_df):,} resim")

print(f"\n📁 Çıktı dosyaları:")
print(f"   {OUTPUT_DIR}/")
print(f"   ├── golden_dataset.parquet  (folder_name, image_name, embedding, selection_method)")
print(f"   ├── golden_summary.json     (istatistikler)")
print(f"   └── umap_visualization.html (interaktif görselleştirme)")

print(f"\n🔍 Parquet dosyasını okumak için:")
print(f"   df = pd.read_parquet('{output_file}')")
print(f"   embeddings = np.array(df['embedding'].tolist())")

print(f"\n🎯 Sonraki adımlar:")
print(f"   1. UMAP görselleştirmesini kontrol et - dağılım homojen mi?")
print(f"   2. Golden dataset'teki resimleri gözle incele")
print(f"   3. YOLO için etiketleme işlemine başla")

In [ ]:
# 🔍 Golden Dataset'i kontrol et
golden_check = pd.read_parquet(output_file)

print("📊 Golden Dataset içeriği:")
print(f"   Satır sayısı: {len(golden_check):,}")
print(f"   Kolonlar: {list(golden_check.columns)}")

print(f"\n🎯 Seçim methodu dağılımı:")
print(golden_check['selection_method'].value_counts())

print(f"\n🔍 Örnek satırlar:")
for idx, row in golden_check.sample(5).iterrows():
    print(f"   [{row['selection_method']:6}] {row['folder_name']}/{row['image_name'][:40]}...")